# Lecture 14 — Data Formatting
### BTDS331 — Introduction to Data Science | Module 2: Data Collection and Preprocessing

**CO:** CO2 &nbsp;&nbsp;|&nbsp;&nbsp; **Bloom's Level:** K2 (Understand)

This notebook is a **teaching companion** for Lecture 14. For every concept and Pandas method
in the lecture, you get:

1. **What it is** — a one-line definition
2. **Why we use it** — the purpose / real-world reason
3. **Code** — a runnable example, using small sample data built directly in the notebook
   (no external file needed — every cell is self-contained)

### Previous Lecture Recap — Missing Values and Outliers
- Missing values are unavailable or unrecorded values
- `isna()` / `isnull()` detect missing values
- `dropna()` removes missing observations
- `fillna()` replaces missing values
- Outliers are unusually extreme observations
- IQR is commonly used to identify potential outliers
- Outliers should be investigated before removal

**Today's Focus:** Making data consistent and analysis-ready.

### Learning Objectives
After today's lecture, you should be able to:
1. Understand data formatting
2. Identify inconsistent formats
3. Standardize text values
4. Convert numerical data types
5. Convert dates into a consistent format
6. Handle common formatting problems using Pandas
7. Prepare a dataset for further analysis


In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.2


## 1. What is Data Formatting?

**Definition:** Data Formatting is the process of converting data into a consistent and
suitable structure or representation so that it can be correctly analyzed and processed.

**Example:**

| Before | After |
|---|---|
| `" Delhi "` | `Delhi` |
| `"delhi"` | `Delhi` |
| `"DELHI"` | `Delhi` |

**Goal:** Same meaning → Same representation.


## 2. Why is Data Formatting Important?

> ⚠️ Inconsistent data can produce incorrect analysis.

**Example:** `Bangalore`, `Bengaluru`, `bangalore`, `BANGALORE` all mean the same city — but a
computer treats every one of these as a **completely different value**, since string
comparison is exact and case-sensitive by default.

**Problems this causes:**
- Incorrect counting
- Wrong grouping
- Failed comparisons
- Incorrect calculations
- Poor visualizations
- Errors in Machine Learning


## 3. Common Formatting Problems

| Problem | Example |
|---|---|
| Extra spaces | `" Rahul "` |
| Different letter case | `"DELHI"` / `"delhi"` |
| Mixed data types | `25` , `"25"` |
| Different date formats | `05/08/2026` , `2026-08-05` |
| Currency symbols | `₹50,000` |
| Commas in numbers | `"60,000"` |
| Different category names | `"Bengaluru"` / `"Bangalore"` |
| Inconsistent text | `"Male"` / `"M"` |


## 4. The Formatting Workflow

```
Raw Data → Inspect Values → Identify Inconsistencies → Standardize Format
   → Convert Data Types → Validate Results → Clean Dataset
```

**Key Principle:** Format → Validate → Analyze


## 5. Text Formatting with Pandas

Pandas gives every text (string) column a `.str` accessor with vectorized string methods —
these apply the same operation to every value in the column at once.

### a) Remove Extra Spaces — `.str.strip()`
**What it is:** Removes leading and trailing whitespace from each string.
**Why we use it:** Leading/trailing spaces are invisible in a spreadsheet but make
`" Rahul"` and `"Rahul"` compare as **different** values to a computer.
```python
df["Name"] = df["Name"].str.strip()
```

### b) Convert to Lowercase — `.str.lower()`
**What it is:** Converts every character to lowercase.
**Why we use it:** A common step before comparing or standardizing text values so that case
differences don't cause mismatches.
```python
df["Name"] = df["Name"].str.lower()
```

### c) Convert to Uppercase — `.str.upper()`
**What it is:** Converts every character to uppercase.
**Why we use it:** Useful for codes/IDs that are conventionally displayed in uppercase
(e.g. country codes, product codes).
```python
df["Name"] = df["Name"].str.upper()
```


In [3]:
names = pd.Series([" Rahul ", "PRIYA", "aman", " Neha "])
print("Original: ", names.tolist())

print("strip():  ", names.str.strip().tolist())
print("lower():  ", names.str.strip().str.lower().tolist())
print("upper():  ", names.str.strip().str.upper().tolist())

Original:  [' Rahul ', 'PRIYA', 'aman', ' Neha ']
strip():   ['Rahul', 'PRIYA', 'aman', 'Neha']
lower():   ['rahul', 'priya', 'aman', 'neha']
upper():   ['RAHUL', 'PRIYA', 'AMAN', 'NEHA']


## 6. Standardizing Text — combining `.str.strip()` with `.str.title()`

**What it is:** `.str.title()` capitalizes the first letter of every word ("Title Case").
Chaining `.str.strip().str.title()` is a very common one-line pattern to standardize
free-text fields.

**Why we use it:** This turns `" Delhi "`, `"delhi"`, `"DELHI"`, `"Delhi"` — four different
representations of the exact same value — into a single consistent form: `Delhi`.

```python
df["City"] = (
    df["City"]
    .str.strip()
    .str.title()
)
```


In [4]:
city = pd.Series([" Delhi ", "delhi", "DELHI", "Delhi"])
print("Before:", city.tolist())

city_clean = city.str.strip().str.title()
print("After: ", city_clean.tolist())
print("Unique values now:", city_clean.nunique())   # 4 -> 1

Before: [' Delhi ', 'delhi', 'DELHI', 'Delhi']
After:  ['Delhi', 'Delhi', 'Delhi', 'Delhi']
Unique values now: 1


## 7. Replacing Inconsistent Values — `.replace()`

**What it is:** `.replace()` maps specific values to a chosen standard value, using a
dictionary of `{old_value: new_value}` pairs.

**Why we use it:** `.str.title()` fixes case and spacing, but it **cannot** fix genuinely
*different spellings* of the same thing — e.g. "Bangalore" vs "Bengaluru" are two different
words for the same city, so no amount of case-conversion will merge them. `.replace()` lets
you explicitly map every known variant to one standard label.

```python
df["City"] = df["City"].replace({
    "Bengaluru": "Bangalore",
    "BANGALORE": "Bangalore",
    "bangalore": "Bangalore"
})
```
> 💡 **Tip:** Use `.replace()` to map all inconsistent values to a single standard value.


In [5]:
city2 = pd.Series(["Bangalore", "Bengaluru", "BANGALORE", "bangalore"])
print("Before:", city2.tolist())

city2_clean = city2.replace({
    "Bengaluru": "Bangalore",
    "BANGALORE": "Bangalore",
    "bangalore": "Bangalore"
})
print("After: ", city2_clean.tolist())

Before: ['Bangalore', 'Bengaluru', 'BANGALORE', 'bangalore']
After:  ['Bangalore', 'Bangalore', 'Bangalore', 'Bangalore']


## 8. Converting Numeric Data — Removing Commas & Symbols

**What it is:** Numbers are sometimes stored as **text (strings)** because they contain
non-numeric characters like commas (`"60,000"`) or currency symbols (`"₹80000"`). Before
converting to a real numeric type, these characters must be stripped out.

**Why we use it:** Pandas/Python cannot do arithmetic on a string. `"60,000"` looks like a
number to a human but is just text to the computer — you need to remove the comma first.

```python
df["Salary"] = df["Salary"].str.replace(",", "", regex=False)
```


In [6]:
salary = pd.Series(["50000", "60000", "70,000", "₹80000"])
print("Before:", salary.tolist())

# Step 1: remove commas
salary_step1 = salary.str.replace(",", "", regex=False)
# Step 2: remove the currency symbol (₹)
salary_step2 = salary_step1.str.replace("₹", "", regex=False)
print("After removing commas & symbol:", salary_step2.tolist())

Before: ['50000', '60000', '70,000', '₹80000']
After removing commas & symbol: ['50000', '60000', '70000', '80000']


## 9. `pd.to_numeric()` — Converting to Numeric Data Types

**What it is:** A function that converts a column's values into a proper numeric dtype
(`int64` or `float64`). The `errors="coerce"` argument tells pandas: *"if a value can't be
converted, don't crash — just turn it into `NaN`."*

**Why we use it:** After removing commas/symbols, values are still stored as **text**. This is
the step that turns them into **real numbers** you can do maths on — and safely flags any
leftover invalid entries (like `"unknown"`) as missing data instead of crashing your whole
script.

```python
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
```

**Example:**

| Input Value | Output (Numeric) |
|---|---|
| `"85"` | `85` |
| `"92"` | `92` |
| `"unknown"` | `NaN` |


In [7]:
marks = pd.Series(["85", "92", "unknown", "78"])

marks_numeric = pd.to_numeric(marks, errors="coerce")
print(marks_numeric)
print("dtype:", marks_numeric.dtype)

0    85.0
1    92.0
2     NaN
3    78.0
dtype: float64
dtype: float64


In [8]:
# Full numeric-conversion pipeline: comma removal + symbol removal + to_numeric
salary_final = pd.to_numeric(salary_step2, errors="coerce")
print(salary_final)
print("dtype:", salary_final.dtype)

0    50000
1    60000
2    70000
3    80000
dtype: int64
dtype: int64


## 10. Checking Data Types — `df.dtypes`

**What it is:** Returns the data type of every column in a DataFrame (recap from Lecture 12,
essential here too).

**Why we use it:** The very first check before formatting — it tells you exactly which columns
are stored as the *wrong* type (e.g. a numeric-looking `Salary` column stored as `object`/text)
so you know exactly what needs converting.


In [9]:
example_df = pd.DataFrame({
    "Name": ["John", "Mary", "Alice"],
    "Age": [25, 30, 22],
    "Salary": ["50000", "60000", "70000"],   # numeric, but stored as text (object)
    "Marks": [85.5, 91.0, 72.3]
})

print("BEFORE conversion:")
print(example_df.dtypes)

BEFORE conversion:
Name          str
Age         int64
Salary        str
Marks     float64
dtype: object


## 11. Data Type Conversion — `astype()` vs `pd.to_numeric()`

**What it is:** Two ways to change a column's data type:

- **`.astype("int64")` / `.astype("float64")`** — a direct, strict type cast. Fast, but throws
  an error if even one value can't be converted.
- **`pd.to_numeric(..., errors="coerce")`** — a safer conversion that turns unconvertible
  values into `NaN` instead of crashing.

**Why we use each:**
- Use `.astype()` when you're confident every value is already clean and convertible.
- Use `pd.to_numeric(errors="coerce")` when the column might still contain invalid/text
  entries you want surfaced as missing values.

```python
df["Age"] = df["Age"].astype("int64")                 # Integer
df["Marks"] = df["Marks"].astype("float64")            # Float
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")   # Numeric (safe)
```
> ⚠️ **Important:** Only convert when the values are compatible with the target data type.


In [10]:
example_df["Age"] = example_df["Age"].astype("int64")
example_df["Marks"] = example_df["Marks"].astype("float64")
example_df["Salary"] = pd.to_numeric(example_df["Salary"], errors="coerce")

print("AFTER conversion:")
print(example_df.dtypes)
example_df

AFTER conversion:
Name          str
Age         int64
Salary      int64
Marks     float64
dtype: object


,Name,Age,Salary,Marks
0,John,25,50000,85.5
1,Mary,30,60000,91.0
2,Alice,22,70000,72.3


## 12. Date Formatting

Dates often arrive in many different, inconsistent formats:

- `05/08/2026`
- `2026-08-05`
- `05-Aug-2026`
- `August 5, 2026`

**Why this is a problem:** You can't sort, filter, or calculate date differences reliably
while dates are stored as plain text in mixed formats.

**For analysis, convert them all to a consistent `datetime` representation** using
`pd.to_datetime()`.

```python
df["Date"] = pd.to_datetime(df["Date"])
```

> Pandas provides `to_datetime()` for parsing many date representations automatically.


## 14. Extracting Date Components — the `.dt` accessor

**What it is:** Once a column is `datetime64[ns]`, the `.dt` accessor unlocks year, month,
day, hour, minute, second, weekday, and more, as separate values.

**Why we use it:** Very common in analysis — e.g. grouping sales by `Month`, or filtering
records to a specific `Year`.

```python
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"]   = df["Date"].dt.day
```


## 15. Formatting Dates for Display — `.dt.strftime()`

**What it is:** `strftime()` ("string format time") converts a `datetime` value back into a
custom **string** for display, using format codes.

**Why we use it:** A `datetime64` object is great for calculations, but reports, exports, and
user interfaces usually need a specific, human-readable text format (e.g. `DD-MM-YYYY`).

```python
df["Date"].dt.strftime("%d-%m-%Y")
```

**Common Format Codes:**

| Code | Meaning |
|---|---|
| `%Y` | Year |
| `%m` | Month |
| `%d` | Day |
| `%H` | Hour |
| `%M` | Minute |
| `%S` | Second |


In [14]:
d = pd.to_datetime(pd.Series(["2026-08-05"]))
print("datetime64 value:  ", d.iloc[0])
print("strftime('%d-%m-%Y'):", d.dt.strftime("%d-%m-%Y").iloc[0])
print("strftime('%B %d, %Y'):", d.dt.strftime("%B %d, %Y").iloc[0])

datetime64 value:   2026-08-05 00:00:00
strftime('%d-%m-%Y'): 05-08-2026
strftime('%B %d, %Y'): August 05, 2026


## 16. Categorical Data Formatting

**What it is:** Categorical/label columns (like Gender, Status, Category) often contain the
same category written in several different ways: full words, abbreviations, different casing.

**Why we use it:** If "Male", "M", and "male" are all treated as different categories, your
groupings, counts, and charts will all be wrong.

**Two-step fix (matches the lecture):**

**Step 1 — Standardize:** strip spaces + lowercase everything
```python
df["Gender"] = (
    df["Gender"]
    .str.strip()
    .str.lower()
)
```

**Step 2 — Replace short forms:** expand abbreviations to full, consistent labels
```python
df["Gender"] = df["Gender"].replace({
    "m": "male",
    "f": "female"
})
```

> 💡 **Best Practice:** Always standardize categorical values for consistency in analysis and
> reporting.


In [16]:
student_df = pd.DataFrame({
    "Name": [" Rahul ", "PRIYA", " Aman "],
    "City": ["delhi", "Delhi", "DELHI"],
    "Marks": ["85", "90", "unknown"],
    "Date": ["05/08/2026", "2026-08-06", "07/08/2026"]
})

print("RAW DATA:")
student_df

RAW DATA:


,Name,City,Marks,Date
0,Rahul,delhi,85,05/08/2026
1,PRIYA,Delhi,90,2026-08-06
2,Aman,DELHI,unknown,07/08/2026


## 18. Practical Cleaning Code

**What it is:** The complete formatting pipeline applied together, column by column — exactly
as shown in the lecture.

**Why we use it:** This is the reusable template you'll apply to almost every raw dataset
before analysis.

```python
import pandas as pd

df["Name"] = (
    df["Name"]
    .str.strip()
    .str.title()
)

df["City"] = (
    df["City"]
    .str.strip()
    .str.title()
)

df["Marks"] = pd.to_numeric(
    df["Marks"],
    errors="coerce"
)

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)
```


## 19. Before vs After Formatting

| Raw Data | Formatted Data |
|---|---|
| `" Rahul "` | `Rahul` |
| `"PRIYA"` | `Priya` |
| `"delhi"` | `Delhi` |
| `"85"` | `85` |
| `"unknown"` | `NaN` |
| `05/08/2026` | `2026-05-08` * |

> ⚠️ **Important:** Date interpretation depends on the input format. Always verify whether the
> source uses `DD/MM/YYYY` or `MM/DD/YYYY` **before** conversion — `pd.to_datetime()` may guess
> incorrectly if the format is ambiguous (e.g. `05/08/2026` could mean 5-Aug or 8-May). Use the
> explicit `format=` argument when you know the exact source format, to avoid silent
> misinterpretation.

```python
# Being explicit about the source format avoids ambiguity:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
```


In [19]:
# Demonstrating the ambiguity, and the fix with an explicit format
ambiguous_date = pd.Series(["05/08/2026"])

# Pandas' default guess (may interpret as MM/DD/YYYY depending on version/locale):
print("Default parse:", pd.to_datetime(ambiguous_date, errors="coerce").iloc[0])

# Being explicit that the source is DD/MM/YYYY:
print("Explicit DD/MM/YYYY:", pd.to_datetime(ambiguous_date, format="%d/%m/%Y", errors="coerce").iloc[0])

Default parse: 2026-05-08 00:00:00
Explicit DD/MM/YYYY: 2026-08-05 00:00:00


## 20. Data Validation After Formatting

> ⚠️ Formatting is **not complete** until the result is checked.

**Check with:**
```python
df.info()
df.dtypes
df.head()
df.isna().sum()
```

**Validate that:**
- ✅ Correct data types
- ✅ Expected categories
- ✅ Correct date interpretation
- ✅ No unexpected values
- ✅ Missing values identified

> 💡 **Best Practice:** Always validate after formatting to ensure accuracy and reliability of
> the cleaned data.


## 21. Real-World Example: E-Commerce Data

**Raw Product Data**

| Product | Price | City | Order Date |
|---|---|---|---|
| Laptop | `₹60,000` | delhi | `05/08/2026` |
| Laptop | `60000` | Delhi | `2026-08-06` |
| Mobile | `₹25,000` | DELHI | `07/08/2026` |

**Formatting Tasks:**
1. Remove currency symbols
2. Remove commas
3. Convert price to numeric
4. Standardize city names
5. Convert dates to datetime

**Result:** Data becomes suitable for **Sales Analysis → Visualization → Prediction**.


## 24. Key Takeaways

1. Data formatting makes representations consistent.
2. Text values may require trimming and case standardization.
3. Numeric values should use appropriate numeric data types.
4. `pd.to_numeric()` converts values to numeric format.
5. `pd.to_datetime()` converts date values into datetime format.
6. Date formatting should consider the original date convention.
7. Categorical values should use consistent labels.
8. Always validate the dataset after formatting.


## 25. Quick Reference Table

| Method | What it does | When to use |
|---|---|---|
| `.str.strip()` | Removes leading/trailing spaces | Text columns with stray spaces |
| `.str.lower()` / `.str.upper()` | Converts case | Standardizing before comparison |
| `.str.title()` | Capitalizes first letter of each word | Standardizing names/places |
| `.replace({old: new})` | Maps specific values to a standard value | Fixing genuinely different spellings |
| `.str.replace(",", "", regex=False)` | Removes a specific character | Cleaning commas/symbols from numbers |
| `pd.to_numeric(col, errors="coerce")` | Converts text to numeric, invalid → NaN | Safely converting numeric-looking text |
| `.astype("int64")` / `.astype("float64")` | Strict type cast | When data is already clean |
| `pd.to_datetime(col, errors="coerce")` | Converts text to datetime, invalid → NaT | Standardizing date columns |
| `.dt.year` / `.dt.month` / `.dt.day` | Extracts date components | Grouping/filtering by date parts |
| `.dt.strftime("%d-%m-%Y")` | Formats datetime back to a display string | Reports, exports, UI |
| `df.dtypes` | Shows the type of every column | Before and after formatting |
